# Turn Multi-Sensor Visualizer

Visualization-only notebook for one turn folder at a time, with `right`, `left`, `trunk`, and `sacrum` synchronized CSV files. It uses the same turn-window and step-count ideas as `turn_validation.ipynb`, but plots one selected turn activity across all four sensors.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "imu_features").exists():
    REPO_ROOT = Path("..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks

from imu_features.config import PipelineConfig
from imu_features.utils import (
    detect_threshold_turn_window,
    estimate_sampling_interval_s,
    preprocess_activity_dataframe,
    robust_p2p_threshold,
    select_motion_acc_signal,
    select_turn_angular_signal,
    window_peak_to_peak,
)
from validation_plots import shared_validation as sv

CONFIG = PipelineConfig(dataset_root="Data_Sensors")
sv.configure(CONFIG, sv.plot_params_from_config(CONFIG))

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True


def visible_sensor_items(sensor_data):
    return [(sensor, item) for sensor, item in sensor_data.items() if item is not None and SHOW_SENSOR.get(sensor, True)]

print(f"Setup complete | turn_threshold_k={CONFIG.window_gate.turn_threshold_k} | p2p_cap_scale={CONFIG.window_gate.p2p_cap_scale}")


In [ ]:
LAST_TURN_DIR_PATH = REPO_ROOT / ".data_sensors_turn_last_dir"
DEFAULT_TURN_ROOT = REPO_ROOT / "Data_Sensors" / "Converted Files_synchronized" / "NonStroke01" / "Visit1" / "TurnL"

# Default is False so rerunning the notebook does not create a Tkinter window.
# Set True only when you intentionally want to choose a new turn folder.
USE_TKINTER_FOLDER_DIALOG = True

# Optional: paste a TurnL or TurnR folder path here to avoid Tkinter completely.
MANUAL_TURN_FOLDER = ""


def _last_turn_dir(default_dir=DEFAULT_TURN_ROOT):
    try:
        if LAST_TURN_DIR_PATH.exists():
            cached = Path(LAST_TURN_DIR_PATH.read_text().strip()).expanduser()
            if cached.exists():
                return cached
    except OSError:
        pass
    return Path(default_dir)


def _remember_turn_dir(selected_dir):
    try:
        LAST_TURN_DIR_PATH.write_text(str(Path(selected_dir).resolve()))
    except OSError:
        pass


def choose_turn_folder(initial_dir=None):
    import tkinter as tk
    from tkinter import filedialog

    start_dir = _last_turn_dir(initial_dir or DEFAULT_TURN_ROOT)
    root = None
    selected = ""
    try:
        root = tk.Tk()
        root.title("Choose Turn Folder")
        root.resizable(False, False)
        root.attributes("-topmost", True)

        width, height = 260, 80
        root.update_idletasks()
        screen_w = root.winfo_screenwidth()
        screen_h = root.winfo_screenheight()
        x = max(0, int((screen_w - width) / 2))
        y = max(0, int((screen_h - height) / 2))
        root.geometry(f"{width}x{height}+{x}+{y}")

        tk.Label(root, text="Opening folder picker...", padx=20, pady=20).pack(expand=True, fill="both")
        root.lift()
        root.focus_force()
        root.update()

        selected = filedialog.askdirectory(
            parent=root,
            title="Choose a TurnL or TurnR folder containing right/left/trunk/sacrum",
            initialdir=str(start_dir),
        )
    finally:
        if root is not None:
            try:
                root.withdraw()
                root.update_idletasks()
                root.update()
                root.quit()
                root.destroy()
            except tk.TclError:
                pass
            root = None

    if selected:
        selected = Path(selected).resolve()
        _remember_turn_dir(selected)
        return selected
    return None


manual = str(MANUAL_TURN_FOLDER).strip()
if manual:
    TURN_FOLDER = Path(manual).expanduser().resolve()
    _remember_turn_dir(TURN_FOLDER)
elif USE_TKINTER_FOLDER_DIALOG:
    TURN_FOLDER = choose_turn_folder()
else:
    TURN_FOLDER = _last_turn_dir(DEFAULT_TURN_ROOT)

TURN_ACTIVITY = "left_turn" if "turnl" in str(TURN_FOLDER).lower() or "left" in str(TURN_FOLDER).lower() else "right_turn"
TURN_LABEL = "left_turn" if TURN_ACTIVITY == "left_turn" else "right_turn"

print("Selected folder:", TURN_FOLDER)
print("Turn activity:", TURN_LABEL)
print("Tkinter opens only when USE_TKINTER_FOLDER_DIALOG=True.")


In [ ]:
SENSOR_ORDER = ["right", "left", "trunk", "sacrum"]
SHOW_RIGHT = True
SHOW_LEFT = True
SHOW_TRUNK = True
SHOW_SACRUM = True
SHOW_SENSOR = {
    "right": SHOW_RIGHT,
    "left": SHOW_LEFT,
    "trunk": SHOW_TRUNK,
    "sacrum": SHOW_SACRUM,
}

TURN_STEP_SMOOTH_SIGMA = 1.0
TURN_STEP_THRESHOLD_K = float(sv.PLOT_PARAMS.get("turn_compare_threshold_k", 1.0))
TURN_STEP_MIN_DISTANCE_S = 0.37


def find_sensor_csv(turn_folder, sensor_name):
    turn_folder = Path(turn_folder)
    direct_dir = turn_folder / sensor_name
    if direct_dir.exists():
        csvs = sorted(direct_dir.glob("*synchronized*.csv")) or sorted(direct_dir.glob("*.csv"))
        if csvs:
            return csvs[0]
    matches = []
    for path in turn_folder.rglob("*.csv"):
        if sensor_name.lower() in [part.lower() for part in path.parts]:
            sync_rank = 0 if "synchronized" in path.name.lower() else 1
            matches.append((sync_rank, len(path.parts), path.name.lower(), path))
    if matches:
        return sorted(matches)[0][-1]
    return None


def _close_boolean_gaps(mask, max_gap_samples):
    mask = np.asarray(mask, dtype=bool)
    if len(mask) == 0 or max_gap_samples <= 0:
        return mask
    out = mask.copy()
    i = 0
    while i < len(out):
        if out[i]:
            i += 1
            continue
        j = i
        while j < len(out) and not out[j]:
            j += 1
        if i > 0 and j < len(out) and out[i - 1] and out[j] and (j - i) <= max_gap_samples:
            out[i:j] = True
        i = j
    return out


def refine_turn_window_to_main_motion(time_s, angular_abs, threshold, fs_hz, fallback_mask):
    time_s = np.asarray(time_s, dtype=float)
    angular_abs = np.asarray(angular_abs, dtype=float)
    fallback_mask = np.asarray(fallback_mask, dtype=bool)
    if len(time_s) == 0 or len(angular_abs) != len(time_s):
        return np.nan, np.nan, np.nan, np.zeros(len(time_s), dtype=bool)
    if not np.isfinite(threshold):
        threshold = np.nan

    active = np.isfinite(angular_abs) & np.isfinite(threshold) & (angular_abs >= threshold)
    if np.isfinite(fs_hz) and fs_hz > 0:
        active = _close_boolean_gaps(active, max_gap_samples=max(1, int(round(0.15 * fs_hz))))
        min_run_samples = max(1, int(round(0.20 * fs_hz)))
        pad_samples = max(1, int(round(0.15 * fs_hz)))
    else:
        min_run_samples = 1
        pad_samples = 1

    active_values = angular_abs[active & np.isfinite(angular_abs)]
    if len(active_values) == 0:
        return np.nan, np.nan, np.nan, fallback_mask
    strong_floor = max(float(threshold), 0.15 * float(np.nanmax(active_values)))

    strong_runs = []
    i = 0
    while i < len(active):
        if not active[i]:
            i += 1
            continue
        j = i
        while j < len(active) and active[j]:
            j += 1
        run_values = angular_abs[i:j]
        if (j - i) >= min_run_samples and np.isfinite(run_values).any() and float(np.nanmax(run_values)) >= strong_floor:
            strong_runs.append((i, j - 1))
        i = j

    if not strong_runs:
        if np.any(fallback_mask):
            idx = np.where(fallback_mask)[0]
            return float(time_s[idx[0]]), float(time_s[idx[-1]]), float(time_s[idx[-1]] - time_s[idx[0]]), fallback_mask
        return np.nan, np.nan, np.nan, np.zeros(len(time_s), dtype=bool)

    start_idx = max(0, strong_runs[0][0] - pad_samples)
    end_idx = min(len(time_s) - 1, strong_runs[-1][1] + pad_samples)
    mask = np.zeros(len(time_s), dtype=bool)
    mask[start_idx:end_idx + 1] = True
    return float(time_s[start_idx]), float(time_s[end_idx]), float(time_s[end_idx] - time_s[start_idx]), mask


def turn_summary_from_full_signal(df, meta):
    t = df["time_s"].to_numpy(dtype=float)
    angular_signal, gyro_source = select_turn_angular_signal(df, CONFIG, meta.fs_hz)
    angular_abs = np.abs(angular_signal)
    turn_window_sec = getattr(CONFIG.window_gate, "turn_window_sec", CONFIG.window_gate.window_sec)
    turn_threshold, turn_pp = robust_p2p_threshold(
        time_s=t,
        signal=angular_abs,
        window_sec=turn_window_sec,
        k=CONFIG.window_gate.turn_threshold_k,
        fallback=CONFIG.window_gate.turn_min_amp_threshold,
        cap_scale=CONFIG.window_gate.p2p_cap_scale,
    )
    raw_start_s, raw_end_s, raw_duration_s, raw_turn_mask = detect_threshold_turn_window(
        angular_signal_abs=angular_abs,
        time_s=t,
        threshold=turn_threshold,
        window_sec=turn_window_sec,
        min_duration_s=CONFIG.window_gate.turn_min_duration_s,
    )
    turn_start_s, turn_end_s, turn_duration_s, turn_mask = refine_turn_window_to_main_motion(
        time_s=t,
        angular_abs=angular_abs,
        threshold=turn_threshold,
        fs_hz=meta.fs_hz,
        fallback_mask=raw_turn_mask,
    )

    acc_signal, acc_source = select_motion_acc_signal(df, CONFIG.prefer_useracc_for_motion)
    t_turn = t[turn_mask] if np.any(turn_mask) else np.array([], dtype=float)
    acc_turn = acc_signal[turn_mask] if np.any(turn_mask) else np.array([], dtype=float)
    acc_turn_smooth = np.array([], dtype=float)
    peak_indices = np.array([], dtype=int)
    peak_times = np.array([], dtype=float)
    peak_values = np.array([], dtype=float)
    step_threshold = np.nan
    fs_turn = 1.0 / estimate_sampling_interval_s(t_turn) if len(t_turn) > 2 else meta.fs_hz
    if len(acc_turn) >= 3 and np.isfinite(fs_turn) and fs_turn > 0:
        acc_turn_smooth = gaussian_filter1d(acc_turn, sigma=TURN_STEP_SMOOTH_SIGMA)
        step_threshold = TURN_STEP_THRESHOLD_K * float(np.nanmean(acc_turn_smooth))
        min_distance = max(1, int(round(TURN_STEP_MIN_DISTANCE_S * fs_turn)))
        peak_indices, _ = find_peaks(acc_turn_smooth, height=step_threshold, distance=min_distance)
        peak_times = t_turn[peak_indices]
        peak_values = acc_turn_smooth[peak_indices]

    gyro_start_s = turn_start_s
    gyro_end_s = turn_end_s
    if len(peak_indices) and len(t_turn) == len(acc_turn_smooth):
        first_peak_idx = int(peak_indices[0])
        last_peak_idx = int(peak_indices[-1])

        below_before = np.where(
            np.isfinite(acc_turn_smooth[: first_peak_idx + 1])
            & np.isfinite(step_threshold)
            & (acc_turn_smooth[: first_peak_idx + 1] < step_threshold)
        )[0]
        if len(below_before):
            start_local_idx = min(int(below_before[-1]) + 1, first_peak_idx)
        else:
            above_before = np.where(
                np.isfinite(acc_turn_smooth[: first_peak_idx + 1])
                & np.isfinite(step_threshold)
                & (acc_turn_smooth[: first_peak_idx + 1] >= step_threshold)
            )[0]
            start_local_idx = int(above_before[0]) if len(above_before) else first_peak_idx
        turn_start_s = float(t_turn[start_local_idx])

        below_after = np.where(
            np.isfinite(acc_turn_smooth[last_peak_idx:])
            & np.isfinite(step_threshold)
            & (acc_turn_smooth[last_peak_idx:] < step_threshold)
        )[0]
        if len(below_after):
            end_local_idx = int(last_peak_idx + below_after[0])
        else:
            end_local_idx = len(t_turn) - 1
        turn_end_s = float(t_turn[end_local_idx])
        turn_duration_s = float(turn_end_s - turn_start_s)
        turn_mask = (t >= turn_start_s) & (t <= turn_end_s) & np.isfinite(angular_abs)

    if np.any(turn_mask):
        ang_for_stats = angular_abs[turn_mask]
    else:
        ang_for_stats = angular_abs

    step_count = float(len(peak_indices)) if len(t_turn) else np.nan
    steps_per_second = step_count / turn_duration_s if np.isfinite(step_count) and np.isfinite(turn_duration_s) and turn_duration_s > 0 else np.nan
    return {
        "start_s": turn_start_s,
        "end_s": turn_end_s,
        "turn_duration_s": turn_duration_s,
        "turn_mask": turn_mask,
        "raw_turn_mask": raw_turn_mask,
        "raw_start_s": raw_start_s,
        "raw_end_s": raw_end_s,
        "gyro_start_s": gyro_start_s,
        "gyro_end_s": gyro_end_s,
        "turn_threshold": turn_threshold,
        "turn_pp": turn_pp,
        "gyro_source": gyro_source,
        "angular_signal": angular_signal,
        "angular_abs": angular_abs,
        "peak_angular_velocity": float(np.nanmax(ang_for_stats)) if np.isfinite(ang_for_stats).any() else np.nan,
        "mean_angular_velocity": float(np.nanmean(ang_for_stats)) if np.isfinite(ang_for_stats).any() else np.nan,
        "step_count": step_count,
        "steps_per_second": steps_per_second,
        "acc_source": acc_source,
        "acc_signal": acc_signal,
        "t_turn": t_turn,
        "acc_turn": acc_turn,
        "acc_turn_smooth": acc_turn_smooth,
        "step_peak_indices": peak_indices,
        "step_peak_times": peak_times,
        "step_peak_values": peak_values,
        "step_threshold": step_threshold,
        "fs_turn": fs_turn,
    }


def load_sensor_recording(turn_folder, sensor_name):
    csv_path = find_sensor_csv(turn_folder, sensor_name)
    if csv_path is None:
        return None
    raw = pd.read_csv(csv_path)
    df, meta = preprocess_activity_dataframe(raw, CONFIG)
    summary = turn_summary_from_full_signal(df, meta)
    return {
        "sensor": sensor_name,
        "path": csv_path,
        "df": df,
        "meta": meta,
        "time_s": df["time_s"].to_numpy(dtype=float),
        "summary": summary,
    }


def load_all_sensors(turn_folder):
    return {sensor: load_sensor_recording(turn_folder, sensor) for sensor in SENSOR_ORDER}


if TURN_FOLDER is None:
    SENSOR_DATA = {}
    print("No folder selected")
else:
    SENSOR_DATA = load_all_sensors(TURN_FOLDER)
    for sensor, item in SENSOR_DATA.items():
        if item is None:
            print(f"[missing] {sensor}")
        else:
            meta = item["meta"]
            s = item["summary"]
            print(f"[loaded] {sensor}: {item['path']} | n={len(item['df'])} | duration={meta.duration_s:.2f}s | axis={s['gyro_source']} | turn={s['turn_duration_s']:.2f}s")


In [ ]:
SHOW_GYRO_X = True
SHOW_GYRO_Y = True
SHOW_GYRO_Z = True
SHOW_GYRO_RESULTANT = True


def plot_sensor_gyroscope(sensor_data):
    items = visible_sensor_items(sensor_data)
    if not items:
        print("No visible sensors. Set one of SHOW_RIGHT/SHOW_LEFT/SHOW_TRUNK/SHOW_SACRUM to True and rerun.")
        return

    gyro_components = [
        ("gyro_x", SHOW_GYRO_X, "x"),
        ("gyro_y", SHOW_GYRO_Y, "y"),
        ("gyro_z", SHOW_GYRO_Z, "z"),
        ("gyro_mag", SHOW_GYRO_RESULTANT, "resultant"),
    ]
    active_components = [(col, label) for col, show, label in gyro_components if show]
    if not active_components:
        print("No gyroscope components selected")
        return

    fig, axes = plt.subplots(len(items), 1, figsize=(12, 3.6 * len(items)), sharex=True)
    if len(items) == 1:
        axes = [axes]

    for ax, (sensor, item) in zip(axes, items):
        df = item["df"]
        t = item["time_s"]
        for col, label in active_components:
            if col in df:
                linewidth = 2.1 if label == "resultant" else 1.25
                color = "black" if label == "resultant" else None
                ax.plot(t, df[col].to_numpy(dtype=float), linewidth=linewidth, color=color, label=label)
        ax.set_title(f"{sensor}: Gyroscope")
        ax.set_ylabel("Angular Velocity")
        ax.legend(loc="upper right", fontsize=8, framealpha=0.85)

    axes[-1].set_xlabel("Time (s)")
    plt.tight_layout()
    plt.show()


plot_sensor_gyroscope(SENSOR_DATA)


In [ ]:
SHOW_ACCEL_X = True
SHOW_ACCEL_Y = True
SHOW_ACCEL_Z = True
SHOW_ACCEL_RESULTANT = True


def plot_sensor_accelerometer(sensor_data):
    items = visible_sensor_items(sensor_data)
    if not items:
        print("No visible sensors. Set one of SHOW_RIGHT/SHOW_LEFT/SHOW_TRUNK/SHOW_SACRUM to True and rerun.")
        return

    accel_components = [
        ("accel_x", SHOW_ACCEL_X, "x"),
        ("accel_y", SHOW_ACCEL_Y, "y"),
        ("accel_z", SHOW_ACCEL_Z, "z"),
        ("acc_mag", SHOW_ACCEL_RESULTANT, "resultant"),
    ]
    active_components = [(col, label) for col, show, label in accel_components if show]
    if not active_components:
        print("No accelerometer components selected")
        return

    fig, axes = plt.subplots(len(items), 1, figsize=(12, 3.6 * len(items)), sharex=True)
    if len(items) == 1:
        axes = [axes]

    for ax, (sensor, item) in zip(axes, items):
        df = item["df"]
        t = item["time_s"]
        for col, label in active_components:
            if col in df:
                linewidth = 2.1 if label == "resultant" else 1.25
                color = "black" if label == "resultant" else None
                ax.plot(t, df[col].to_numpy(dtype=float), linewidth=linewidth, color=color, label=label)
        ax.set_title(f"{sensor}: Accelerometer")
        ax.set_ylabel("Acceleration")
        ax.legend(loc="upper right", fontsize=8, framealpha=0.85)

    axes[-1].set_xlabel("Time (s)")
    plt.tight_layout()
    plt.show()


plot_sensor_accelerometer(SENSOR_DATA)


In [ ]:
def sensor_summary_row(sensor, item):
    s = item["summary"]
    return {
        "sensor": sensor,
        "file": item["path"].name,
        "gyro_axis": s.get("gyro_source"),
        "acc_source": s.get("acc_source"),
        "fs_hz": item["meta"].fs_hz,
        "full_duration_s": item["meta"].duration_s,
        "turn_duration_s": s.get("turn_duration_s", np.nan),
        "start_s": s.get("start_s", np.nan),
        "end_s": s.get("end_s", np.nan),
        "peak_angular_velocity": s.get("peak_angular_velocity", np.nan),
        "mean_angular_velocity": s.get("mean_angular_velocity", np.nan),
        "step_count": s.get("step_count", np.nan),
        "steps_per_second": s.get("steps_per_second", np.nan),
    }


summary_table = pd.DataFrame(
    [sensor_summary_row(sensor, item) for sensor, item in SENSOR_DATA.items() if item is not None]
)
if len(summary_table):
    display(summary_table.round(3))
else:
    print("No sensor data loaded")


In [ ]:
def plot_turn_step_estimates(sensor_data):
    items = visible_sensor_items(sensor_data)
    if not items:
        print("No visible sensors. Set one of SHOW_RIGHT/SHOW_LEFT/SHOW_TRUNK/SHOW_SACRUM to True and rerun.")
        return

    fig, axes = plt.subplots(len(items), 1, figsize=(12, 3.8 * len(items)), sharex=True)
    if len(items) == 1:
        axes = [axes]

    for ax, (sensor, item) in zip(axes, items):
        s = item["summary"]
        t = item["time_s"]
        acc = s["acc_signal"]
        start_s = s.get("start_s", np.nan)
        end_s = s.get("end_s", np.nan)
        duration = s.get("turn_duration_s", np.nan)
        step_count = s.get("step_count", np.nan)
        step_threshold = s.get("step_threshold", np.nan)
        t_turn = s.get("t_turn", np.array([], dtype=float))
        smooth = s.get("acc_turn_smooth", np.array([], dtype=float))
        peak_times = s.get("step_peak_times", np.array([], dtype=float))
        peak_values = s.get("step_peak_values", np.array([], dtype=float))

        ax.plot(t, acc, color="lightsteelblue", linewidth=1.0, alpha=0.75, label=s.get("acc_source", "acc"))
        if len(t_turn) == len(smooth) and len(smooth):
            ax.plot(t_turn, smooth, color="steelblue", linewidth=1.8, label=f"Gaussian smoothed {s.get('acc_source', 'acc')} (sigma={TURN_STEP_SMOOTH_SIGMA:g})")
        if len(peak_times) == len(peak_values) and len(peak_times):
            ax.scatter(peak_times, peak_values, color="crimson", s=38, zorder=5, label=f"peaks={len(peak_times)}")
        if np.isfinite(start_s):
            ax.axvline(start_s, color="green", linestyle="--", linewidth=1.5, label="turn start")
        if np.isfinite(end_s):
            ax.axvline(end_s, color="gray", linestyle="--", linewidth=1.5, label="turn end")
        if np.isfinite(step_threshold):
            ax.axhline(step_threshold, color="firebrick", linestyle="--", linewidth=1.4, label=f"K*mean threshold={step_threshold:.3f}")

        ax.set_title(f"{sensor}: {TURN_LABEL} find_peaks Turn Step Estimate")
        ax.set_ylabel("Acceleration Magnitude")
        ax.text(
            0.01,
            0.96,
            f"start={start_s:.3f}s | end={end_s:.3f}s | duration={duration:.2f}s\nsteps={step_count:.1f} | steps/s={s.get('steps_per_second', np.nan):.3f}",
            transform=ax.transAxes,
            va="top",
            ha="left",
            bbox=dict(facecolor="white", alpha=0.85, edgecolor="none", boxstyle="round,pad=0.25"),
        )
        ax.legend(loc="upper right", fontsize=8, framealpha=0.85)

    axes[-1].set_xlabel("Time (s)")
    plt.tight_layout()
    plt.show()


plot_turn_step_estimates(SENSOR_DATA)
